# EDU Silver schema creation

Creates and validates the 24 empty Silver Delta tables defined directly in this notebook. This notebook is the deployable schema source of truth for workload users; it does not read Bronze data or populate Silver rows. Exact matching tables are skipped without receiving a new Delta version.

The repository model CSVs remain design and review evidence, but they are not uploaded to Fabric or required at runtime.

In [ ]:
# Deployment parameters
allow_destructive_changes = False
exit_notebook_on_success = True

import hashlib
import json
import re

from pyspark.sql import types as T

expected_table_count = 24
expected_column_count = 158
expected_relationship_count = 34
supported_types = {
    "STRING": T.StringType(),
    "DATE": T.DateType(),
    "TIMESTAMP": T.TimestampType(),
    "INT": T.IntegerType(),
    "DOUBLE": T.DoubleType(),
    "BOOLEAN": T.BooleanType(),
}
identifier_pattern = re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")

## Embedded Silver contract

Each column tuple is `(name, type, declared nullable, primary key, foreign key)`. Dictionary order defines table order and tuple order defines physical column order. Foreign keys also define the complete logical relationship set and creation dependencies.

In [ ]:
TABLES = {
    "District": [
        ("district_id", "STRING", False, True, None),
        ("name", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "School": [
        ("school_id", "STRING", False, True, None),
        ("district_id", "STRING", False, False, "District.district_id"),
        ("name", "STRING", False, False, None),
        ("code", "STRING", False, False, None),
        ("school_type", "STRING", False, False, None),
        ("city", "STRING", False, False, None),
        ("language", "STRING", False, False, None),
        ("website", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "Student": [
        ("student_id", "STRING", False, True, None),
        ("given_name", "STRING", False, False, None),
        ("surname", "STRING", False, False, None),
        ("status", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    # The school's postal address, kept apart from School so an existing School table keeps
    # its exact schema. One row per school; the key is also the reference to School.
    "SchoolAddress": [
        ("school_id", "STRING", False, True, "School.school_id"),
        ("street_address", "STRING", False, False, None),
        ("city", "STRING", False, False, None),
        ("state", "STRING", True, False, None),
        ("postal_code", "STRING", True, False, None),
        ("county", "STRING", True, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "Teacher": [
        ("teacher_id", "STRING", False, True, None),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("name", "STRING", False, False, None),
        ("department", "STRING", False, False, None),
        ("role_title", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "Course": [
        ("course_id", "STRING", False, True, None),
        ("name", "STRING", False, False, None),
        ("code", "STRING", False, False, None),
        ("description", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "AcademicTerm": [
        ("term_id", "STRING", False, True, None),
        ("valid_from", "DATE", False, False, None),
        ("valid_to", "DATE", False, False, None),
    ],
    "Section": [
        ("section_id", "STRING", False, True, None),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("course_id", "STRING", False, False, "Course.course_id"),
        ("term_id", "STRING", False, False, "AcademicTerm.term_id"),
        ("grade", "STRING", False, False, None),
        ("name", "STRING", False, False, None),
        ("code", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "SchoolEnrollment": [
        ("enrollment_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("grade", "STRING", False, False, None),
        ("valid_from", "DATE", False, False, None),
        ("valid_to", "DATE", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "SectionEnrollment": [
        ("enrollment_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("section_id", "STRING", False, False, "Section.section_id"),
        ("valid_from", "DATE", False, False, None),
        ("valid_to", "DATE", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "TeacherAssignment": [
        ("assignment_id", "STRING", False, True, None),
        ("teacher_id", "STRING", False, False, "Teacher.teacher_id"),
        ("section_id", "STRING", False, False, "Section.section_id"),
        ("valid_from", "DATE", False, False, None),
        ("valid_to", "DATE", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "SourceIdentity": [
        ("mapping_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("source_name", "STRING", False, False, None),
        ("source_user_id", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "LmsSection": [
        ("mapping_id", "STRING", False, True, None),
        ("section_id", "STRING", False, False, "Section.section_id"),
        ("source_name", "STRING", False, False, None),
        ("source_class_id", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "CalendarDay": [
        ("calendar_id", "STRING", False, True, None),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("instructional_date", "DATE", False, False, None),
        ("expected_minutes", "INT", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "Attendance": [
        ("attendance_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("attendance_date", "DATE", False, False, None),
        ("status", "STRING", False, False, None),
        ("minutes_absent", "INT", True, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "LearningContent": [
        ("content_id", "STRING", False, True, None),
        ("section_id", "STRING", False, False, "Section.section_id"),
        ("name", "STRING", False, False, None),
        ("description", "STRING", False, False, None),
        ("due_at", "TIMESTAMP", True, False, None),
        ("max_score", "DOUBLE", True, False, None),
        ("source_name", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "Obligation": [
        ("obligation_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("content_id", "STRING", False, False, "LearningContent.content_id"),
        ("section_id", "STRING", False, False, "Section.section_id"),
        ("due_at", "TIMESTAMP", True, False, None),
        ("valid_from", "DATE", False, False, None),
        ("valid_to", "DATE", False, False, None),
    ],
    "SubmissionState": [
        ("state_id", "STRING", False, True, None),
        ("obligation_id", "STRING", False, False, "Obligation.obligation_id"),
        ("state", "STRING", False, False, None),
        ("source_missing", "BOOLEAN", True, False, None),
        ("source_late", "BOOLEAN", True, False, None),
        ("source_record_id", "STRING", False, False, None),
        ("source_modified_at", "TIMESTAMP", True, False, None),
    ],
    "Submission": [
        ("submission_id", "STRING", False, True, None),
        ("obligation_id", "STRING", False, False, "Obligation.obligation_id"),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("content_id", "STRING", False, False, "LearningContent.content_id"),
        ("attempt", "INT", False, False, None),
        ("submitted_at", "TIMESTAMP", False, False, None),
        ("score", "DOUBLE", True, False, None),
        ("state", "STRING", False, False, None),
        ("source_late", "BOOLEAN", True, False, None),
        ("source_missing", "BOOLEAN", True, False, None),
        ("source_record_id", "STRING", False, False, None),
        ("source_modified_at", "TIMESTAMP", True, False, None),
    ],
    "AssessmentInstrument": [
        ("assessment_id", "STRING", False, True, None),
        ("name", "STRING", False, False, None),
        ("subject", "STRING", False, False, None),
        ("unit", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "AssessmentWindow": [
        ("window_id", "STRING", False, True, None),
        ("name", "STRING", False, False, None),
        ("start_date", "DATE", False, False, None),
        ("end_date", "DATE", False, False, None),
        ("sequence", "INT", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "AssessmentResult": [
        ("result_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("assessment_id", "STRING", False, False, "AssessmentInstrument.assessment_id"),
        ("window_id", "STRING", False, False, "AssessmentWindow.window_id"),
        ("assessment_date", "DATE", False, False, None),
        ("score", "DOUBLE", True, False, None),
        ("standard_error", "DOUBLE", True, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    # Teacher-logged actions for a learner, such as a check-in or a family contact. The
    # reason names the rule that prompted the action; the outcome note is what the teacher
    # recorded at the follow-up. Both are descriptive and never a risk score.
    "ActionType": [
        ("action_type_id", "STRING", False, True, None),
        ("name", "STRING", False, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
    "InterventionAction": [
        ("action_id", "STRING", False, True, None),
        ("student_id", "STRING", False, False, "Student.student_id"),
        ("school_id", "STRING", False, False, "School.school_id"),
        ("section_id", "STRING", True, False, "Section.section_id"),
        ("teacher_id", "STRING", False, False, "Teacher.teacher_id"),
        ("action_type_id", "STRING", False, False, "ActionType.action_type_id"),
        ("action_date", "DATE", False, False, None),
        ("reason", "STRING", False, False, None),
        ("reason_category", "STRING", False, False, None),
        ("status", "STRING", False, False, None),
        ("follow_up_date", "DATE", True, False, None),
        ("completed_date", "DATE", True, False, None),
        ("outcome_note", "STRING", True, False, None),
        ("source_record_id", "STRING", False, False, None),
    ],
}

## Validate the embedded Silver model

Validate counts, names, keys, foreign-key targets, supported types, and dependency order before inspecting or changing any Lakehouse table.

In [ ]:
try:
    notebookutils.fs.ls("Files")
except Exception as exc:
    raise RuntimeError("Attach the target Silver Lakehouse as the notebook's default Lakehouse") from exc

errors = []
table_order = list(TABLES)
defined_columns = {
    (table_name, column[0])
    for table_name, columns in TABLES.items()
    for column in columns
}
column_count = sum(len(columns) for columns in TABLES.values())

if len(TABLES) != expected_table_count:
    errors.append(f"Expected {expected_table_count} tables, received {len(TABLES)}")
if column_count != expected_column_count:
    errors.append(f"Expected {expected_column_count} columns, received {column_count}")

relationships = set()
dependencies = {table_name: set() for table_name in TABLES}
for table_name, columns in TABLES.items():
    if not identifier_pattern.fullmatch(table_name):
        errors.append(f"Invalid table identifier: {table_name!r}")
    primary_keys = []
    seen_columns = set()
    for column_name, data_type, nullable, primary_key, foreign_key in columns:
        if not identifier_pattern.fullmatch(column_name):
            errors.append(f"Invalid column identifier: {table_name}.{column_name!r}")
        if column_name in seen_columns:
            errors.append(f"Duplicate column definition: {table_name}.{column_name}")
        seen_columns.add(column_name)
        if data_type not in supported_types:
            errors.append(f"Unsupported type {data_type!r} on {table_name}.{column_name}")
        if not isinstance(nullable, bool) or not isinstance(primary_key, bool):
            errors.append(f"Nullability and primary-key flags must be booleans on {table_name}.{column_name}")
        if primary_key:
            primary_keys.append(column_name)
        if foreign_key:
            if foreign_key.count(".") != 1:
                errors.append(f"Malformed foreign key on {table_name}.{column_name}: {foreign_key!r}")
                continue
            target_table, target_column = foreign_key.split(".", 1)
            if (target_table, target_column) not in defined_columns:
                errors.append(f"Unresolved foreign key {table_name}.{column_name} -> {foreign_key}")
            else:
                relationships.add((table_name, column_name, target_table, target_column))
                dependencies[table_name].add(target_table)
    if len(primary_keys) != 1:
        errors.append(f"{table_name} must have exactly one primary key, received {primary_keys}")

if len(relationships) != expected_relationship_count:
    errors.append(f"Expected {expected_relationship_count} relationships, received {len(relationships)}")

creation_groups = []
remaining = set(TABLES)
created = set()
while remaining:
    ready = [name for name in table_order if name in remaining and dependencies[name] <= created]
    if not ready:
        errors.append(f"Cyclic table dependencies: {sorted(remaining)}")
        break
    creation_groups.append(ready)
    created.update(ready)
    remaining.difference_update(ready)

if errors:
    raise ValueError("Silver contract validation failed:\n- " + "\n- ".join(errors))

contract_payload = json.dumps(TABLES, separators=(",", ":"), ensure_ascii=True)
schema_contract_hash = hashlib.sha256(contract_payload.encode("utf-8")).hexdigest()
print(
    f"Validated {len(TABLES)} tables, {column_count} columns and "
    f"{len(relationships)} relationships in {len(creation_groups)} dependency groups"
)

## Preflight existing schemas

Build each contract schema and compare every existing table before creating or rebuilding anything. Populated tables are never rebuilt by this notebook.

In [ ]:
def field_metadata(nullable: bool, primary_key: bool, foreign_key: str | None) -> dict:
    metadata = {
        "primaryKey": primary_key,
        "declaredNullable": nullable,
    }
    if foreign_key:
        metadata["foreignKey"] = foreign_key
    return metadata


def build_schema(columns: list[tuple]) -> T.StructType:
    return T.StructType([
        T.StructField(
            column_name,
            supported_types[data_type],
            nullable,
            field_metadata(nullable, primary_key, foreign_key),
        )
        for column_name, data_type, nullable, primary_key, foreign_key in columns
    ])


def schema_differences(expected: T.StructType, actual: T.StructType) -> list[str]:
    differences = []
    if len(expected.fields) != len(actual.fields):
        differences.append(f"column count expected {len(expected.fields)}, received {len(actual.fields)}")
    for position, (expected_field, actual_field) in enumerate(zip(expected.fields, actual.fields), 1):
        if expected_field.name != actual_field.name:
            differences.append(
                f"column {position} name expected {expected_field.name}, received {actual_field.name}"
            )
        if expected_field.dataType != actual_field.dataType:
            differences.append(
                f"{expected_field.name} type expected {expected_field.dataType.simpleString()}, "
                f"received {actual_field.dataType.simpleString()}"
            )
        if expected_field.metadata != actual_field.metadata:
            differences.append(
                f"{expected_field.name} metadata expected {expected_field.metadata}, "
                f"received {actual_field.metadata}"
            )
    return differences


table_schemas = {name: build_schema(columns) for name, columns in TABLES.items()}
table_actions = {}
drift_errors = []
for table_name in table_order:
    if not spark.catalog.tableExists(table_name):
        table_actions[table_name] = "CREATE"
        continue
    differences = schema_differences(table_schemas[table_name], spark.table(table_name).schema)
    if not differences:
        table_actions[table_name] = "SKIP"
        continue
    has_data = spark.table(table_name).limit(1).count() > 0
    if allow_destructive_changes and not has_data:
        table_actions[table_name] = "REBUILD_EMPTY"
    else:
        reason = "table contains data" if has_data else "destructive changes are disabled"
        drift_errors.append(f"{table_name}: {reason}; {'; '.join(differences)}")

if drift_errors:
    raise ValueError("Silver schema drift detected before table creation:\n- " + "\n- ".join(drift_errors))

display(spark.createDataFrame(
    [(name, table_actions[name], len(table_schemas[name].fields)) for name in table_order],
    ["Table", "Action", "Columns"],
))

## Create missing tables

Create tables in dependency order. A second run skips every matching table and performs no Delta writes.

In [ ]:
results = []
for group_number, group in enumerate(creation_groups, 1):
    for table_name in group:
        action = table_actions[table_name]
        if action == "SKIP":
            results.append((table_name, group_number, "SKIPPED_MATCHING", None))
            continue
        try:
            if action == "REBUILD_EMPTY":
                spark.sql(f"DROP TABLE `{table_name}`")
            (spark.createDataFrame([], table_schemas[table_name])
             .write.format("delta").mode("errorifexists").saveAsTable(table_name))
            status = "REBUILT_EMPTY" if action == "REBUILD_EMPTY" else "CREATED"
            results.append((table_name, group_number, status, None))
        except Exception as exc:
            results.append((table_name, group_number, "FAILED", str(exc)[:4000]))
            raise

result_frame = spark.createDataFrame(
    results,
    T.StructType([
        T.StructField("Table", T.StringType(), False),
        T.StructField("CreationGroup", T.IntegerType(), False),
        T.StructField("Status", T.StringType(), False),
        T.StructField("ErrorMessage", T.StringType(), True),
    ]),
)
display(result_frame.orderBy("CreationGroup", "Table"))

## Verify and return

Re-read all physical schemas and return a compact result suitable for Fabric pipeline orchestration.

In [ ]:
verification_errors = []
for table_name in table_order:
    if not spark.catalog.tableExists(table_name):
        verification_errors.append(f"Missing table after creation: {table_name}")
        continue
    differences = schema_differences(table_schemas[table_name], spark.table(table_name).schema)
    if differences:
        verification_errors.append(f"{table_name}: {'; '.join(differences)}")
if verification_errors:
    raise ValueError("Silver schema verification failed:\n- " + "\n- ".join(verification_errors))

tables_created = sum(1 for row in results if row[2] in {"CREATED", "REBUILT_EMPTY"})
tables_skipped = sum(1 for row in results if row[2] == "SKIPPED_MATCHING")
tables_failed = sum(1 for row in results if row[2] == "FAILED")
result = {
    "status": "SUCCEEDED",
    "tablesExpected": len(TABLES),
    "tablesCreated": tables_created,
    "tablesSkipped": tables_skipped,
    "tablesFailed": tables_failed,
    "columnsValidated": column_count,
    "relationshipsValidated": len(relationships),
    "schemaContractHash": schema_contract_hash,
}
display(spark.createDataFrame([result]))
result_json = json.dumps(result, separators=(",", ":"))
print(result_json)
if exit_notebook_on_success:
    notebookutils.notebook.exit(result_json)